# Moves

A move proposes new positions for the walkers at one temperature. The default is `reddemcee.moves.StretchMove(a=2.0)`. The default stretch move and temperature swaps are described in the [reddemcee paper](https://ui.adsabs.harvard.edu/abs/2026A%26A...706A.323P).

We will sample a correlated 2D Gaussian with a wider Gaussian prior and keep the temperature ladder fixed for both runs.

In [1]:
import numpy as np
import reddemcee
from reddemcee.moves import DifferentialEvolutionMove, StretchMove

def loglike(theta):
    x, y = theta
    return -0.5 * (x**2 + ((y - x) / 0.3)**2)

def logprior(theta):
    return -0.5 * np.sum((theta / 5.)**2)

betas = [1., 0.4, 0.1]
nwalkers, ndim_, nsweeps = 24, 2, 200
np.random.seed(1234)
p0 = np.random.randn(len(betas), nwalkers, ndim_)

## One move

Pass a move instance with `moves=`. For the stretch move, `a` controls the range of stretch factors. Here we use `a=1.5`.

The original stretch move is described in [Goodman & Weare (2010), *Ensemble samplers with affine invariance*, CAMCoS 5(1), 65–80](https://doi.org/10.2140/camcos.2010.5.65).

In [2]:
sampler_stretch = reddemcee.PTSampler(nwalkers, ndim_, loglike, logprior,
                                       betas=betas, adapt_mode='NONE',
                                       moves=StretchMove(a=1.5))
state_stretch = sampler_stretch.run_mcmc(p0, nsweeps=nsweeps)

## Mixing moves

Differential evolution proposes a step along the difference between two complementary walkers. `DifferentialEvolutionMove` uses reddemcee’s tempered acceptance and state updates with [emcee's differential evolution proposal](https://emcee.readthedocs.io/en/stable/user/moves/#emcee.moves.DEMove).

Pass a flat list of `(move, weight)` pairs to mix moves. The sampler normalizes the weights and draws a move at each update of the ensemble at each temperature. Here the probabilities are 80% stretch and 20% differential evolution. A list of move instances, such as `[StretchMove(), DifferentialEvolutionMove()]`, gives each move equal weight.

In [3]:
my_moves = [(StretchMove(a=1.5), 0.8),
            (DifferentialEvolutionMove(), 0.2)]

sampler_mixed = reddemcee.PTSampler(nwalkers, ndim_, loglike, logprior,
                                     betas=betas, adapt_mode='NONE',
                                     moves=my_moves)
state_mixed = sampler_mixed.run_mcmc(p0, nsweeps=nsweeps)

Both runs start at `p0`. We can look at the mean fraction of proposals accepted by the walkers in the cold chain (`beta=1`).

In [4]:
print(f'Stretch: {sampler_stretch.acceptance_fraction[0].mean():.3f}')
print(f'Mixed:   {sampler_mixed.acceptance_fraction[0].mean():.3f}')

Stretch: 0.835
Mixed:   0.745


This move has been implemented natively into `reddemcee` as `DifferentialEvolutionMove`.